# 10 · Data analyst case study: where is the money going?
A stakeholder asks: *"Margin looks fine overall. Is anything costing us more than it should, and what should we do about it?"*

The analyst workflow: **clarify the question → pull the right numbers → find the driver → size the impact → recommend → state the caveats.** You finish with a short memo, which is the real deliverable.

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import numpy as np
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv
df["not_completed"] = (df["status"] != "Completed").astype(int)

## 1. Clarify and define
Before any code, write down the definitions. Here: *margin = total profit ÷ total revenue* (a ratio of sums), and *cost vs budget = average cost per record ÷ the entity's `baseline_target` − 1* (positive means over budget).

## 2. The trend: is margin moving?

In [ ]:
monthly = df.set_index("record_date")[["revenue", "profit"]].resample("MS").sum()
monthly["margin_pct"] = (monthly["profit"] / monthly["revenue"] * 100).round(2)
monthly.round(0).join(monthly["margin_pct"], rsuffix="_")[["revenue", "profit", "margin_pct"]]

## 3. Find the driver: cost vs budget by entity

In [ ]:
ent = df.groupby("entity_name").agg(records=("fact_id", "count"), revenue=("revenue", "sum"), profit=("profit", "sum"),
                                     avg_cost=("operational_cost", "mean"), budget=("baseline_target", "mean"))
ent["cost_vs_budget_pct"] = ((ent["avg_cost"] / ent["budget"] - 1) * 100).round(1)
ent.sort_values("cost_vs_budget_pct", ascending=False).round(1)

### Your turn 1
Which entity is the most over budget? Store its **name** in `worst_entity`.

In [ ]:
worst_entity = None

In [ ]:
check("worst_entity", worst_entity == ent["cost_vs_budget_pct"].idxmax(), "ent['cost_vs_budget_pct'].idxmax()")

### Your turn 2: size the impact
How many dollars did that entity spend **above its budget** over the whole period? That is `SUM(operational_cost − baseline_target)` over its records. Store it in `overspend_usd` (a float).

In [ ]:
overspend_usd = None

In [ ]:
sub = df[df["entity_name"] == ent["cost_vs_budget_pct"].idxmax()]
truth = float((sub["operational_cost"] - sub["baseline_target"]).sum())
check("overspend_usd", overspend_usd is not None and abs(overspend_usd - truth) < 0.01, "filter the entity's rows, then (cost - baseline_target).sum()")
if overspend_usd is not None:
    print(f"overspend = ${overspend_usd:,.0f}  ({overspend_usd / df['revenue'].sum():.2%} of total revenue)")

### Your turn 3: slice by time
Which weekday (`day_name`) has the highest **weighted margin** (total profit ÷ total revenue for that weekday)? Store the day name in `best_margin_weekday`.

In [ ]:
best_margin_weekday = None

In [ ]:
g = df.groupby("day_name")[["profit", "revenue"]].sum()
check("best_margin_weekday", best_margin_weekday == (g["profit"] / g["revenue"]).idxmax(), "sum profit and revenue per weekday first, then divide")

## 4. Write the memo
Double-click this cell and fill it in. Rules of thumb: lead with the answer, give one number that matters, say what you'd do, and say what could make you wrong.

**Headline:** (one sentence, e.g. "Entity X is over budget by __%, costing about $__.")

**Evidence:** (2-3 bullets with numbers from above)

**So what:** (why a manager should care)

**Recommendation:** (a specific action, owner, and how you'd measure success)

**Caveats:** (what the data can't tell us: is the budget realistic? is this one entity's mix of work different?)